In [3]:
from notebookutils import mssparkutils
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


StatementMeta(, abf7d380-5033-453d-b0ba-8ee9061df6b6, 5, Finished, Available, Finished, False)

In [4]:
import time

start = time.time()

print("Running Gold Layer...")

mssparkutils.notebook.run("04_1_gold_fact_review", 600, {"batch_id": batch_id})
mssparkutils.notebook.run("04_2_gold_dim_business", 600, {"batch_id": batch_id})
mssparkutils.notebook.run("04_3_gold_dim_user", 600, {"batch_id": batch_id})
mssparkutils.notebook.run("04_4_gold_dim_date", 600, {"batch_id": batch_id})
mssparkutils.notebook.run("04_5_gold_business_metrics", 600, {"batch_id": batch_id})
mssparkutils.notebook.run("04_6_gold_city_metrics", 600, {"batch_id": batch_id})

print("Gold tables refreshed.")

end = time.time()
print(f"Gold pipeline finished in {round(end-start,2)} seconds")

StatementMeta(, abf7d380-5033-453d-b0ba-8ee9061df6b6, 6, Finished, Available, Finished, False)

Running Gold Layer...


Gold tables refreshed.
Gold pipeline finished in 571.71 seconds


In [6]:
# ── Delta Lake maintenance ─────────────────────────────────────────────────
# Z-Order by most common query dimensions, then compact small files

maintenance_tables = {
    "fact_review": ["business_id"],
    "dim_business": ["business_id"],
    "dim_user": ["user_id"],
    "business_metrics_gold": ["business_id"],
    "city_metrics_gold": ["state", "city"],
}

for table, zorder_cols in maintenance_tables.items():
    cols = ", ".join(zorder_cols)
    print(f"[OPTIMIZE] {table} ZORDER BY ({cols})")
    spark.sql(f"OPTIMIZE yelp_lakehouse.dbo.{table} ZORDER BY {cols}")

# Vacuum: remove files older than 7 days (168 hours)
for table in maintenance_tables:
    print(f"[VACUUM] {table}")
    spark.sql(f"VACUUM yelp_lakehouse.dbo.{table} RETAIN 168 HOURS")

print("[DONE] Delta maintenance complete")

StatementMeta(, abf7d380-5033-453d-b0ba-8ee9061df6b6, 8, Finished, Available, Finished, False)

[OPTIMIZE] fact_review ZORDER BY (business_id)
[OPTIMIZE] dim_business ZORDER BY (business_id)
[OPTIMIZE] dim_user ZORDER BY (user_id)
[OPTIMIZE] business_metrics_gold ZORDER BY (business_id)
[OPTIMIZE] city_metrics_gold ZORDER BY (state, city)
[VACUUM] fact_review
[VACUUM] dim_business
[VACUUM] dim_user
[VACUUM] business_metrics_gold
[VACUUM] city_metrics_gold
[DONE] Delta maintenance complete
